# Train YOLO26 on full football frames

This notebook uses the 512-image Roboflow v7 export and its existing train, valid and test splits. It copies full images to a new prepared dataset and converts polygon annotations to bounding boxes; the export is untouched. It starts from pretrained YOLO26m and uses the RX 6800 XT through DirectML.

The supplied images are 1920 × 1080. YOLO letterboxes them to the chosen training size while preserving their proportions. Training uses random color and geometric augmentation; validation and test are not augmented. The frames come from one video, so results may be optimistic for a new match.

Select the `.venv-directml` kernel (see README), restart it, and run the cells in order.


In [1]:
import sys
print("Notebook Python:", sys.executable)
from pathlib import Path
import json, math, re, shutil, hashlib
from collections import defaultdict
from datetime import datetime
import yaml, torch
from ultralytics import YOLO
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'requirements-yolo.txt').exists()), None)
if ROOT is None: raise RuntimeError('Start Jupyter from the repository or notebooks folder.')
DATASET_DIR = ROOT / 'data/Football Players and Ball.v7i.yolo26'
SOURCE_YAML = DATASET_DIR / 'data.yaml'
MODEL = str(ROOT / 'notebooks' / 'yolo26m.pt') if (ROOT / 'notebooks' / 'yolo26m.pt').is_file() else 'yolo26m.pt'  # Downloaded on first use if absent.
IMAGE_SIZE = 1280  # Full 1920x1080 images are letterboxed without stretching.
EPOCHS, BATCH, WORKERS, PATIENCE = 100, 1, 0, 25  # Batch 1 saves GPU memory at high resolution.
# DirectML runs the model on the RX 6800 XT; validation and loss use the CPU.
sys.path.insert(0, str(ROOT / 'scripts'))
from yolo_directml import select_amd_device, directml_training
DEVICE = select_amd_device()
CONFIDENCE, NMS_IOU, SEED = 0.10, 0.70, 42
print('Repository:', ROOT)


Notebook Python: c:\Users\liamw\Documents\Neural-Net-Ninjas\.venv-directml\Scripts\python.exe
DirectML training device: AMD Radeon RX 6800 XT (privateuseone:0)
Repository: c:\Users\liamw\Documents\Neural-Net-Ninjas


## 1. Inspect the export and hardware

DirectML is a separate GPU backend, so `torch.cuda.is_available()` will be false even when the RX 6800 XT is working. The selected DirectML device is printed above. The experimental bridge uses CPU loss and validation; GPU memory reporting is unavailable. The YAML class names define your labels; the COCO baseline detection head will be adapted to these classes.


In [2]:
print('PyTorch:', torch.__version__, '| HIP/ROCm:', torch.version.hip, '| CUDA/ROCm available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
if not SOURCE_YAML.is_file(): raise FileNotFoundError(f'Unzip the Roboflow export and set DATASET_DIR. Missing: {SOURCE_YAML}')
config = yaml.safe_load(SOURCE_YAML.read_text(encoding='utf-8'))
raw = config.get('names')
if isinstance(raw, list): names = raw
elif isinstance(raw, dict): names = [raw.get(i, raw.get(str(i))) for i in range(len(raw))]
else: raise ValueError('data.yaml must have a names list or integer-indexed mapping.')
if not names or any(not isinstance(n,str) or not n.strip() for n in names): raise ValueError('Invalid class names.')
images = sorted(p for p in DATASET_DIR.rglob('*') if p.is_file() and p.parent.name.lower() == 'images' and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp'})
print('Classes:', names, '| Images:', len(images))
for image in images[:20]: print(image.relative_to(DATASET_DIR))
if not images: raise ValueError('No images found in export images folders.')


PyTorch: 2.4.1+cpu | HIP/ROCm: None | CUDA/ROCm available: False
Classes: ['player', 'soccer ball'] | Images: 512
test\images\121364_0_frame_036_jpg.rf.79735d3af5120614357a39cce849f3c3.jpg
test\images\121364_0_frame_104_jpg.rf.dab86317d678ae8d6995e0607597bf7d.jpg
test\images\121364_0_frame_111_jpg.rf.978f47409321173434442d242f398a31.jpg
test\images\121364_0_frame_137_jpg.rf.1aecff71be2212853c8624954e74d7af.jpg
test\images\121364_0_frame_144_jpg.rf.1a7447d9c4edc19bdb396e6796eeda9d.jpg
test\images\121364_0_frame_169_jpg.rf.82484de6d6e2e9e81c4cc4a8fb7fc427.jpg
test\images\121364_0_frame_178_jpg.rf.82cf3a125a48621a7b8a4b61442f2118.jpg
test\images\121364_0_frame_185_jpg.rf.08440189dd88f13de61cb9cae334d015.jpg
test\images\121364_0_frame_186_jpg.rf.263ec2e2a1d5e87e139a1ef2183c64de.jpg
test\images\121364_0_frame_199_jpg.rf.0e5308ad3137ea710589b82598f8c84d.jpg
test\images\121364_0_frame_205_jpg.rf.1f7385c5ff8a1dfb8dd5a5a1ec00a25b.jpg
test\images\121364_0_frame_220_jpg.rf.c1129d10bf7970b5c8010ff

## 2. Validate labels and prepare full images

The export's existing train, validation and test splits are preserved. No tiles or grid are created. Polygon labels are converted to enclosing bounding boxes in a prepared copy; box labels are retained. The notebook checks for missing labels, duplicate image bytes and frame overlap between splits. Consecutive frames can still look alike across splits, so use separate footage for a stronger final test.


In [3]:
def read_labels(image):
    label = image.parent.parent / 'labels' / (image.stem + '.txt')
    if not label.is_file(): raise FileNotFoundError(f'Missing label: {label}; create an empty file only for a confirmed negative.')
    boxes = []
    for number, line in enumerate(label.read_text(encoding='utf-8').splitlines(),1):
        try:
            values = list(map(float, line.split()))
            if not values or not all(math.isfinite(v) for v in values): raise ValueError('missing or non-finite value')
            cls, points = values[0], values[1:]
            if not cls.is_integer() or not 0 <= cls < len(names): raise ValueError('invalid class ID')
            if len(points) == 4:
                xc, yc, w, h = points
                if not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < w <= 1 and 0 < h <= 1 and xc-w/2 >= -1e-6 and xc+w/2 <= 1+1e-6 and yc-h/2 >= -1e-6 and yc+h/2 <= 1+1e-6): raise ValueError('invalid normalized box')
                boxes.append(f'{int(cls)} {xc:.8f} {yc:.8f} {w:.8f} {h:.8f}')
                continue
            if len(points) < 6 or len(points) % 2: raise ValueError('expected bounding box or polygon with at least 3 coordinate pairs')
            if any(v < 0 or v > 1 for v in points): raise ValueError('polygon coordinate outside tile')
            xs, ys = points[0::2], points[1::2]
            x1,x2,y1,y2 = min(xs),max(xs),min(ys),max(ys)
            if x2 <= x1 or y2 <= y1: raise ValueError('polygon has zero width or height')
            boxes.append(f"{int(cls)} {(x1+x2)/2:.8f} {(y1+y2)/2:.8f} {x2-x1:.8f} {y2-y1:.8f}")
        except ValueError as exc: raise ValueError(f'{label}:{number}: {exc}') from exc
    return boxes

from collections import Counter
split_names = {'train': 'train', 'valid': 'val', 'test': 'test'}
splits = {name: [] for name in split_names.values()}
digests, frame_splits, class_counts = {}, {}, Counter()
for image in images:
    source_split = image.parent.parent.name.lower()
    if source_split not in split_names:
        raise ValueError(f'Unexpected split folder: {image}')
    group = split_names[source_split]
    boxes = read_labels(image)
    class_counts.update(int(box.split()[0]) for box in boxes)
    digest = hashlib.sha256(image.read_bytes()).hexdigest()
    if digest in digests:
        raise ValueError(f'Duplicate image bytes: {image} and {digests[digest]}')
    digests[digest] = image
    match = re.fullmatch(r'(?P<sequence>.+)_frame_(?P<frame>\d+)_jpg\.rf\.[0-9a-f]{32}', image.stem)
    if not match:
        raise ValueError(f'Cannot identify source frame: {image.name}')
    frame = (match.group('sequence'), int(match.group('frame')))
    if frame in frame_splits and frame_splits[frame] != group:
        raise ValueError(f'Frame {frame} appears in multiple splits.')
    frame_splits[frame] = group
    splits[group].append(image)
for group, items in splits.items():
    if not items:
        raise ValueError(f'No images in {group} split')
    print(f'{group}: {len(items)} full images')
print('Annotations by class:', {names[i]: class_counts[i] for i in range(len(names))})
print('Review class counts and split sizes before training.')


train: 362 full images
val: 102 full images
test: 48 full images
Annotations by class: {'player': 11113, 'soccer ball': 264}
Review class counts and split sizes before training.


In [4]:
# Copy full images and converted detection labels; leave the original export untouched.
output = ROOT / 'runs' / 'training-data' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
output.mkdir(parents=True, exist_ok=False)
manifest = {}
for group, items in splits.items():
    image_out, label_out = output/'images'/group, output/'labels'/group
    image_out.mkdir(parents=True); label_out.mkdir(parents=True)
    manifest[group] = []
    for index, image in enumerate(items):
        target = f'image_{index:06d}'
        shutil.copy2(image, image_out/(target+image.suffix.lower()))
        boxes = read_labels(image)
        (label_out/(target+'.txt')).write_text('\n'.join(boxes) + ('\n' if boxes else ''), encoding='utf-8')
        manifest[group].append({'source': str(image.relative_to(DATASET_DIR)), 'output': target})
    print(group, ': copied', len(manifest[group]), 'full images')
(output/'split_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
dataset_yaml = output/'dataset.yaml'
dataset_yaml.write_text(yaml.safe_dump({'path':str(output.resolve()), 'train':'images/train', 'val':'images/val', 'test':'images/test', 'names':names}, sort_keys=False), encoding='utf-8')
print('Prepared:', dataset_yaml)


train : copied 362 full images
val : copied 102 full images
test : copied 48 full images
Prepared: c:\Users\liamw\Documents\Neural-Net-Ninjas\runs\training-data\20261113_181958_154581\dataset.yaml


## 3. Train the baseline

The DirectML bridge runs the model on the GPU and loss and validation on the CPU. The pretrained detection head is adapted to the export's classes. Training applies random hue, saturation, brightness, translation, scale and horizontal flips; validation and test are not augmented. Mosaic and mixup are disabled. Start at 1280 pixels and batch 1. If GPU memory is insufficient, reduce the size to 960 or try YOLO26s. If training is stable and you want to test throughput, try batch 2.


In [5]:
model = YOLO(MODEL)  # Standard YOLO26 baseline; pretrained weights may download on first use.
with directml_training(DEVICE):
    results = model.train(data=str(dataset_yaml), epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH,
    device='cpu', amp=False, workers=WORKERS, patience=PATIENCE, seed=SEED,
    project=str(ROOT/'runs'/'local-training'), name='football', exist_ok=False,
    fliplr=0.5, flipud=0.0, hsv_h=0.01, hsv_s=0.25, hsv_v=0.20,
    degrees=0.0, translate=0.05, scale=0.15, mosaic=0.0, mixup=0.0)
best = Path(results.save_dir)/'weights'/'best.pt'
if not best.is_file(): raise FileNotFoundError(f'No best checkpoint found: {best}')
print('Best checkpoint:', best)


engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\liamw\Documents\Neural-Net-Ninjas\runs\training-data\20261113_181958_154581\dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.01, hsv_s=0.25, hsv_v=0.2, imgsz=1280, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=c:\Users\liamw\Documents\Neural-Net-Ninjas\notebooks\yolo26m.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=football-2, nbs=6

c:\Users\liamw\Documents\Neural-Net-Ninjas\.venv-directml\Lib\site-packages\torch\optim\adamw.py:529: UserWarning: The operator 'aten::lerp.Scalar_out' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  torch._foreach_lerp_(device_exp_avgs, device_grads, 1 - beta1)


      1/100         0G     0.9044      2.329  0.0008497         23       1280: 100% ━━━━━━━━━━━━ 362/362 2.8it/s 2:070.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 1.1it/s 47.1s0.9ss
                   all        102       2263      0.958      0.871      0.939      0.567

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      2/100         0G     0.9533     0.6915  0.0009461         22       1280: 100% ━━━━━━━━━━━━ 362/362 3.3it/s 1:490.3sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 1.1it/s 46.6s0.9ss
                   all        102       2263      0.906      0.921      0.933      0.561

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      3/100         0G     0.9751     0.4947  0.0009166         21       1280: 100% ━━━━━━━━━━━━ 362/362 3.4it/s 1:470.3sss
                 Class     

## 4. Evaluate and inspect

Evaluation runs on the CPU. Validation is for choosing confidence, NMS IoU and model settings. `iou` here is the overlap threshold for non-maximum suppression, **not** the IoU used to report mAP@50 or mAP@50–95. Higher NMS IoU retains more overlapping predictions, potentially increasing duplicates; confidence often has a more direct effect on recall. YOLO26 may use an end-to-end NMS-free inference path, in which case changing NMS IoU has no effect. Check your installed Ultralytics version and prediction mode. Test is held out until you finish tuning. Inspect per-class ball recall, precision and confusion matrix, not just overall mAP.


In [6]:
best_model = YOLO(str(best))
val_metrics = best_model.val(data=str(dataset_yaml), split='val', imgsz=IMAGE_SIZE, batch=BATCH,
                             device='cpu', workers=WORKERS, conf=CONFIDENCE, iou=NMS_IOU,
                             project=str(ROOT / 'runs' / 'local-training'), name='validation', exist_ok=False)
print('Validation mAP50:', val_metrics.box.map50, 'mAP50-95:', val_metrics.box.map)
print('Per-class AP50:', val_metrics.box.ap50)


Ultralytics 8.4.212  Python-3.12.6 torch-2.4.1+cpu CPU (12th Gen Intel Core i7-12700K)
YOLO26m summary (fused): 130 layers, 20,350,994 parameters, 0 gradients, 68.1 GFLOPs
val: Fast image access  (ping: 0.1±0.0 ms, read: 859.2±167.4 MB/s, size: 100.2 KB)
val: Scanning C:\Users\liamw\Documents\Neural-Net-Ninjas\runs\training-data\20261113_181958_154581\labels\val.cache... 102 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 102/102  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 102/102 3.8it/s 26.9s0.3s
                   all        102       2263      0.992      0.973      0.967      0.726
                player        102       2213      0.986      0.986      0.979      0.853
           soccer ball         50         50      0.999       0.96      0.955        0.6
Speed: 0.6ms preprocess, 252.1ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to C:\Users\liamw\Documents\Neural-Net-Ninjas\runs\local-

In [7]:
# Run only after deciding settings from validation. This reports full-image metrics.
test_metrics = best_model.val(data=str(dataset_yaml), split='test', imgsz=IMAGE_SIZE, batch=BATCH,
                              device='cpu', workers=WORKERS, conf=CONFIDENCE, iou=NMS_IOU,
                              project=str(ROOT / 'runs' / 'local-training'), name='test', exist_ok=False)
print('Test mAP50:', test_metrics.box.map50, 'mAP50-95:', test_metrics.box.map)
print('Per-class AP50:', test_metrics.box.ap50)
print('Checkpoint to integrate after review:', best)


Ultralytics 8.4.212  Python-3.12.6 torch-2.4.1+cpu CPU (12th Gen Intel Core i7-12700K)
YOLO26m summary (fused): 130 layers, 20,350,994 parameters, 0 gradients, 68.1 GFLOPs
WARNING val: Slow image access detected (ping: 0.1±0.0 ms, read: 21.9±4.7 MB/s, size: 97.3 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning C:\Users\liamw\Documents\Neural-Net-Ninjas\runs\training-data\20261113_181958_154581\labels\test... 48 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 48/48 687.2it/s 0.1s
val: New cache created: C:\Users\liamw\Documents\Neural-Net-Ninjas\runs\training-data\20261113_181958_154581\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 3.6it/s 13.5s0.3s
                   all         48       1064      0.976      0.956      0.964      0.677
                player         48       1041      0.998   

Review `results.csv`, curves and confusion matrices in the run directory. Keep the split manifest and document the baseline, hardware, settings and results. This notebook does not replace the frontend model automatically.
